In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

# Fix for draw_geometries crashing on Wayland
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [ ]:
import open3d as o3d
from pathlib import Path
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from torchvision.utils import make_grid
import optuna
import warnings

from nerf import LNeRF
from ingp import LInstantNGP
from utils.rays import intrinsic, create_rays, look_at, equidistance_rotations

COMPUTE_DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    COMPUTE_DEVICE = torch.device('cuda:0')
elif torch.mps.is_available():
    COMPUTE_DEVICE = torch.device('mps')
print(f"{COMPUTE_DEVICE=}")

RUN_OPT = False

In [ ]:
print("Available objects:")
print("\n".join([p.stem for p in Path(f'../data/raw_objects/').iterdir() if p.is_dir()]))

# NeRF to point cloud evaluation

## Setup

In [ ]:
def get_origin_direction_eq_angles(n, img_shape, focal):
    phis, thetas = equidistance_rotations(n)

    origin, direction = [], []
    for theta, phi in zip(thetas, phis):
        o, d = create_rays(
            img_shape[0],
            img_shape[1],
            intrinsic(focal, img_shape),
            look_at(4, theta, phi)
        )
        origin.append(o)
        direction.append(d)

    return torch.stack(origin), torch.stack(direction)

In [ ]:
MODEL_CHKPT_PATH = Path(
    "../lightning_logs/ingp_weisshai_shark800x800/checkpoints/best_val_psnr_epoch=4.ckpt"
).resolve()
OBJ_NAME = "Weisshai_Great_White_Shark"

hparams_path = (MODEL_CHKPT_PATH / ".." / ".." / "hparams.yaml").resolve()

model = LInstantNGP.load_from_checkpoint(MODEL_CHKPT_PATH, map_location=COMPUTE_DEVICE, hparams_file=hparams_path)
model.freeze()
model.eval()
focal = torch.tensor(model.hparams.focal, dtype=torch.float32, device=COMPUTE_DEVICE)

origin, direction = get_origin_direction_eq_angles(8, (800, 800), focal.expand(2))
print(f"{origin.shape=} {direction.shape=}")

## Naive depth

In [ ]:
shape = origin.shape[:-1]
dl = DataLoader(TensorDataset(origin.flatten(0, -2), direction.flatten(0, -2)), batch_size=2**10)

nv_depth, sp_mask = [], []
with torch.no_grad():
    for o, d in dl:
        rgba, d, _ = model.render_rays(o.to(model.device), d.to(model.device))
        nv_depth.append(d.cpu())
        
nv_depth = torch.cat(nv_depth, 0).reshape(shape + (1,))
print(f"{nv_depth.shape=}")

In [ ]:
nv_depth_mask = ((nv_depth > model.hparams.near) & (nv_depth <= model.hparams.far)).squeeze(-1)
nv_points = origin[nv_depth_mask] + nv_depth[nv_depth_mask] * direction[nv_depth_mask]

nv_points.shape

## Surface Point depth

In [ ]:
shape = origin.shape[:-1]
dl = DataLoader(TensorDataset(origin.flatten(0, -2), direction.flatten(0, -2)), batch_size=2**19, shuffle=False)

rm_depth, sp_mask = [], []
for o, d in dl:
    d, sm = model.locate_density_gradient_based_surface_depth(
        o, d,
        sigma_limit=5.912066757, # sigma_limit=5.912066757,  # 0.01 / sqrt(3) * 1024
        gamma=8e-10,  # gamma=8e-10,
        non_grad_step_size=0.0016532793661392217, # non_grad_step_size=0.001691456,  # sqrt(3) / 1024
        min_step_size=1 / 1024,  # min_step_size=1 / 1024,  # 2 / (2 * sample count)
        max_iters=2**12,  # max_iters=2**12,  # 2*2048 as non_grad_step_size requires a min of 2048 steps from near to far
    )
    d, sm = d.detach().cpu(), sm.cpu()
    rm_depth.append(d)
    sp_mask.append(sm)

rm_depth, sp_mask = torch.cat(rm_depth, 0).reshape(shape + (1,)), torch.cat(sp_mask, 0).reshape(shape)
print(f"{rm_depth.shape=} {sp_mask.shape=}")

In [ ]:
depth_imgs = (rm_depth.expand((-1, -1, -1, 3)) - model.hparams.near) / (model.hparams.far - model.hparams.near)
depth_grid = make_grid(depth_imgs.permute(0, 3, 1, 2), 4).permute(1, 2, 0).clip(0,1)

fig, ax = plt.subplots(1, 1, figsize=(18, 6))
ax.imshow(depth_grid)
ax.axis('off')
plt.show()

In [ ]:
rm_depth_mask = ((rm_depth > model.hparams.near) & (rm_depth <= model.hparams.far)).squeeze(-1)
rm_points = origin[rm_depth_mask] + rm_depth[rm_depth_mask] * direction[rm_depth_mask]

rm_points.shape, rm_points[sp_mask[rm_depth_mask]].shape

## Chamfer Distance

In [ ]:
cloud_from_tensor = lambda tens: o3d.geometry.PointCloud(o3d.utility.Vector3dVector(tens.cpu()))

def load_and_normalize_mesh(obj_path: Path) -> o3d.geometry.TriangleMesh:
    mesh = o3d.io.read_triangle_mesh(obj_path)
    bbox = mesh.get_axis_aligned_bounding_box()
    # Re-centering and scaling to -1:1 bounding box, confirmed same transforms to Mitsuba version
    scale = 1 / max((bbox.get_max_bound() - bbox.get_min_bound()) / 2)
    translate = -bbox.get_center()

    mesh = mesh.translate(translate).scale(scale.item(), [0, 0, 0])
    return mesh

def mesh_to_cloud_signed_distances(mesh: o3d.geometry.TriangleMesh, cloud: o3d.geometry.PointCloud) -> np.ndarray:
    cloud = o3d.t.geometry.PointCloud.from_legacy(cloud)
    mesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
    scene = o3d.t.geometry.RaycastingScene()
    _ = scene.add_triangles(mesh)
    sdf = scene.compute_signed_distance(cloud.point.positions).abs()
    return sdf.numpy()

In [ ]:
mesh = load_and_normalize_mesh(Path(f"../data/raw_objects/{OBJ_NAME}/meshes/model.obj"))
point_cloud = cloud_from_tensor(rm_points)

print(mesh, point_cloud, sep='\n')

In [ ]:
def colored_boxplot(vals, colormap, norm, title, xlim_to_norm_vmax=False):
    # Create horizontal boxplot
    fig, ax = plt.subplots(figsize=(14, 2))
    bp = ax.boxplot(vals, vert=False, widths=1.0)

    for element in ['boxes', 'whiskers', 'caps']:
        plt.setp(bp[element], color="lime", linewidth=1.5)
    plt.setp(bp['fliers'], color="lime", linewidth=1, markeredgecolor="lime")
    plt.setp(bp['medians'], color="white", linewidth=3)

    # Color the background based on distance distribution
    xmin, xmax = ax.get_xlim()
    y_min, y_max = ax.get_ylim()

    # Create a gradient background
    gradient = np.linspace(xmin, xmax, 1024)
    gradient = np.vstack((gradient, gradient))
    extent = (xmin, xmax, y_min - 1, y_max + 1)  # Extend beyond boxplot bounds

    # Apply the colormap to the background
    ax.imshow(
        gradient, 
        aspect='auto', 
        extent=extent, 
        cmap=colormap, 
        alpha=1.0, 
        origin='lower',
        norm=norm
    )

    if xlim_to_norm_vmax:
        dst = norm.vmax - norm.vmin
        ax.set_xlim(norm.vmin - dst * 0.05, norm.vmax + dst * 0.05)

    # Add mean line
    mean = vals.mean()
    if ax.get_xlim()[0] < mean < ax.get_xlim()[1]:
        ax.axvline(mean, color="black", linestyle='--', linewidth=2)
        ax.text(mean, 2.0, f'Mean: {mean:.4f}', ha='center', color="black", fontdict={'weight': 'bold'})

    ax.set_yticks([])  # Hide y-axis as it's just one boxplot
    ax.set_xlabel(title)
    plt.tight_layout()
    plt.show()


In [ ]:
bbox_mask = (rm_points.abs() <= 1.0).all(-1)
#point_cloud = cloud_from_tensor(rm_points[bbox_mask])
point_cloud = cloud_from_tensor(rm_points[sp_mask[rm_depth_mask] & bbox_mask])

#bbox_mask = (nv_points.abs() <= 1.0).all(-1)
#point_cloud = cloud_from_tensor(nv_points[bbox_mask])

# Remove outliers, results of numerical errors and local maxima above threshold in specific points
point_cloud, _ = point_cloud.remove_statistical_outlier(nb_neighbors=30, std_ratio=5.0)

print(point_cloud)
mesh = load_and_normalize_mesh(Path(f"../data/raw_objects/{OBJ_NAME}/meshes/model.obj"))
cd = mesh_to_cloud_signed_distances(mesh, point_cloud)

colormap = plt.get_cmap('seismic')
norm = TwoSlopeNorm(vmin=cd.min(), vcenter=np.quantile(cd, 0.75), vmax=cd.max())
colored_boxplot(cd, colormap, norm, "Chamfer Distance")

rgb = colormap(norm(cd))[:, :3]
point_cloud.colors = o3d.utility.Vector3dVector(rgb)

o3d.visualization.draw_geometries([
    point_cloud
])

In [ ]:
bbox_mask = (rm_points.abs() <= 1.0).all(-1)
rm_point_cloud = cloud_from_tensor(rm_points[sp_mask[rm_depth_mask] & bbox_mask])
rm_point_cloud, _ = rm_point_cloud.remove_statistical_outlier(nb_neighbors=30, std_ratio=5.0)
rm_point_cloud.paint_uniform_color([0.1, 1.0, 0.1])
rm_point_cloud = rm_point_cloud.voxel_down_sample(0.05)

bbox_mask = (nv_points.abs() <= 1.0).all(-1)
nv_point_cloud = cloud_from_tensor(nv_points[bbox_mask])
nv_point_cloud, _ = nv_point_cloud.remove_statistical_outlier(nb_neighbors=30, std_ratio=5.0)
nv_point_cloud.paint_uniform_color([1.0, 0.1, 0.1])
nv_point_cloud = nv_point_cloud.voxel_down_sample(0.05)

mesh = load_and_normalize_mesh(Path(f"../data/raw_objects/{OBJ_NAME}/meshes/model.obj"))
mesh.compute_vertex_normals()

o3d.visualization.draw_geometries([
    rm_point_cloud, nv_point_cloud, mesh
], mesh_show_back_face=True)

# Sigma value colormap

In [ ]:
with torch.no_grad():
    sigmas = model.nerf(rm_points.to(model.device), direction[rm_depth_mask].to(model.device), skip_colors=True)
    sigmas = sigmas.cpu().squeeze(-1)

bbox_mask = (rm_points.abs() <= 1.0).all(-1)
point_cloud = cloud_from_tensor(rm_points[bbox_mask])
print(point_cloud)

m_sigmas = sigmas[bbox_mask]
colormap = plt.get_cmap('RdYlGn')
norm = TwoSlopeNorm(vmin=m_sigmas.min(), vcenter=np.quantile(m_sigmas, 0.25), vmax=np.quantile(m_sigmas, 0.75))
colored_boxplot(m_sigmas, colormap, norm, "Sigma", xlim_to_norm_vmax=True)

rgb = colormap(norm(m_sigmas))[:, :3]
point_cloud.colors = o3d.utility.Vector3dVector(rgb)

o3d.visualization.draw_geometries([
    point_cloud
])

# Poisson Surface Reconstruction

In [ ]:
bbox_mask = (rm_points.abs() <= 1.0).all(-1)
point_cloud = cloud_from_tensor(rm_points[bbox_mask])

normals = []
normal_points = DataLoader(rm_points[bbox_mask], batch_size=2**19, shuffle=False)
for nps in normal_points:
    normals.append(model.estimate_normals(nps).cpu())
normals = torch.cat(normals, dim=0)

point_cloud.normals = o3d.utility.Vector3dVector(normals)

mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(point_cloud, depth=8)
mesh.paint_uniform_color([0.5,0.5,0.5])
mesh.compute_vertex_normals()

mesh_smp = mesh.filter_smooth_laplacian(number_of_iterations=30)
mesh_smp.compute_vertex_normals()
mesh_smp.normalize_normals()

print(mesh, mesh_smp)
o3d.visualization.draw_geometries([mesh_smp])

# Parameter searching

In [ ]:
def raymeshnerf_cloud_mean_chamfer_distance(trial: optuna.Trial):
    ckpt_path = Path(
        "../lightning_logs/ingp_weisshai_shark800x800/checkpoints/best_val_psnr_epoch=4.ckpt"
    ).resolve()
    hparams_path = (MODEL_CHKPT_PATH / ".." / ".." / "hparams.yaml").resolve() 
    
    model = LInstantNGP.load_from_checkpoint(
        ckpt_path,
        map_location=COMPUTE_DEVICE,
        hparams_file=hparams_path
    )
    model.freeze()
    model.eval()
    focal = torch.tensor(model.hparams.focal, dtype=torch.float32, device=COMPUTE_DEVICE)
    
    origin, direction = get_origin_direction_eq_angles(8, (800, 800), focal.expand(2))

    gamma = trial.suggest_float("gamma", 1e-10, 1e-8)
    min_step_size = trial.suggest_float("min_step_size", 1e-5, 1e-3)

    shape = origin.shape[:-1]
    dl = DataLoader(TensorDataset(origin.flatten(0, -2), direction.flatten(0, -2)), batch_size=2**19)
    depth, sp_mask = [], []
    for o, d in dl:
        d, sm = model.locate_density_gradient_based_surface_depth(
            o, d,
            sigma_limit=5.912066757, # sigma_limit=5.912066757,  # 0.01 / sqrt(3) * 1024
            gamma=gamma,
            non_grad_step_size=0.0016532793661392217,  # non_grad_step_size=0.001691456,  # sqrt(3) / 1024
            min_step_size=min_step_size,
            max_iters=2**12,  # 2 * 2048
        )
        d, sm = d.detach().cpu(), sm.cpu()
        depth.append(d)
        sp_mask.append(sm)

    depth, sp_mask = torch.cat(depth, 0).reshape(shape + (1,)), torch.cat(sp_mask, 0).reshape(shape)

    depth_mask = ((depth > model.hparams.near) & (depth <= model.hparams.far)).squeeze(-1)
    points = origin[depth_mask] + depth[depth_mask] * direction[depth_mask]

    # Point cloud including every point (filtered by depth mask)
    point_cloud = cloud_from_tensor(points)
    # Remove obvious outliers, results of numerical errors and local maxima above threshold in specific points
    point_cloud, _ = point_cloud.remove_statistical_outlier(nb_neighbors=30, std_ratio=5.0)

    # Point cloud while only using points that are considered surface points (grad < grad_epsilon)
    sp_point_cloud = cloud_from_tensor(points[sp_mask[depth_mask]])
    # Remove obvious outliers, results of numerical errors and local maxima above threshold in specific points
    sp_point_cloud, _ = sp_point_cloud.remove_statistical_outlier(nb_neighbors=30, std_ratio=5.0)

    if points.numel() == 0 or points[sp_mask[depth_mask]].numel() == 0:
        raise optuna.TrialPruned()

    # Setting verbosity level to Error as mesh doesn't have texture next to it and loads as multiple materials,
    #   this gives warnings but doesn't matter for eval 
    o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)
    mesh = load_and_normalize_mesh(Path("../data/raw_objects/Weisshai_Great_White_Shark/meshes/model.obj").resolve())

    cd = mesh_to_cloud_signed_distances(mesh, point_cloud)
    sp_cd = mesh_to_cloud_signed_distances(mesh, sp_point_cloud)
    # Reset verbosity level to recieve info about potential other problems
    o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Info)

    trial.set_user_attr("All Point Count", len(point_cloud.points))
    # In order: Mean CD All Points, Mean CD Surface Points, Surface Points
    return cd.mean().item(), sp_cd.mean().item(), int(sp_mask[depth_mask].sum().item())


if RUN_OPT:
    study = optuna.create_study(
        study_name="800SharkMinStepGamma",
        directions=["minimize", "minimize", "maximize"],
        storage="sqlite:///ChamferDistance.db",
        load_if_exists=True
    )
    study.set_metric_names([
        "Chamfer Distance | All Points",
        "Chamfer Distance | Surface Points",
        "Surface Point Count"
    ])

    study.optimize(
        raymeshnerf_cloud_mean_chamfer_distance,
        n_trials=20,
        gc_after_trial=True
    )